# PagedCore: does one FP16 value explain the final drift?

Enable **Internet** and **T4 GPU**, then **Run All** in a fresh Kaggle session.
The exact source, frozen setup, retained failed logits, target tensor and export
are embedded. No dataset, Git push or previous session is required. Only GPU 0 is used.

Keep the same 15-token prompt plus one decode step. The previous trace found the
first FP16 difference in the **second decoder layer (index 1), head 21, channel 60**.
The experiment has two arms with fresh caches and the same weights and inputs:

- **Sham:** clone the attention output and write back its original scalar.
  It must reproduce the retained baseline logits and all 440 stage comparisons.
- **Single-scalar intervention:** perform the same clone/write, replacing only
  that scalar with the captured contiguous-reference FP16 value. Every other
  element must remain exact, and all stages before the target must match the sham.

The four cells install/verify, reproduce the original failure, run both arms,
and export one ZIP. The run retains both complete layer traces, full logits,
K/V checks, unchanged history, exact local writes, positions and cleanup.
Numerical failures are results, not reasons to discard the experiment.

**How to interpret it:** if the final error shrinks, the first value contributes
to the drift. If later FP16 differences remain, this value does not explain all
of it. More error is also a valid observation. Inspect the first remaining
difference and all control flags before drawing a conclusion. Two arms on this
one input do not establish broader model accuracy or a general error bound.

This is a diagnostic override in a separate script. It never changes serving
code, thresholds or the comparator, and an intervention passing is not a P5.3 pass.
Download the ZIP even on failure and return it for analysis. If its link fails,
use Kaggle's Output file panel. No whole-suite rerun is needed for this experiment.


In [ ]:
import base64
import hashlib
import io
import json
import os
import subprocess
import sys
import tempfile
import traceback
import zipfile
from pathlib import Path

from IPython.display import FileLink, display

PAYLOAD = "..."
SNAPSHOT_SHA256 = "89082244cbcad581575f98a50a956445533112f45d8cbc7d89635371b67838b4"


output_override = os.environ.get("PAGEDCORE_EVIDENCE_WORKDIR")
working = (
    Path(output_override).expanduser().resolve()
    if output_override
    else Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path.cwd()
)
working.mkdir(parents=True, exist_ok=True)
run_root = Path(tempfile.mkdtemp(prefix="pagedcore-scalar-control-", dir=working))
source = run_root / "source"
evidence = run_root / "evidence"
evidence.mkdir()
payload = base64.b64decode(PAYLOAD)
assert hashlib.sha256(payload).hexdigest() == SNAPSHOT_SHA256
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(source)
identity = json.loads((source / "source-identity.json").read_text())
environment = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES="0",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    TOKENIZERS_PARALLELISM="false",
)
commands = []


def run(name, command, *, required=False):
    record = {"name": name, "argv": list(map(str, command)), "cwd": str(source)}
    commands.append(record)
    print(name, flush=True)
    with (evidence / (name + ".log")).open("w") as log:
        process = subprocess.Popen(
            command, cwd=source, env=environment, stdout=log, stderr=subprocess.STDOUT
        )
        try:
            while process.poll() is None:
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    print(name + " is running; progress is saved in " + str(log.name), flush=True)
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            record["interrupted"] = True
            raise
    record["returncode"] = process.returncode
    (evidence / "commands.json").write_text(json.dumps(commands, indent=2))
    print(name, "exit", process.returncode, flush=True)
    if required and process.returncode:
        raise RuntimeError(name + " failed; inspect its retained log")


def verify_source():
    checks = {
        name: hashlib.sha256((source / name).read_bytes()).hexdigest()
        for name in identity["sha256"]
    }
    assert checks == identity["sha256"], "source changed after notebook generation"
    return checks


setup_ok = False
try:
    verify_source()
    run("setup-uv", [sys.executable, "-m", "pip", "install", "uv==0.11.27"], required=True)
    run(
        "setup-frozen",
        [
            sys.executable,
            "-m",
            "uv",
            "sync",
            "--frozen",
            "--extra",
            "cuda",
            "--extra",
            "dev",
            "--python",
            (source / ".python-version").read_text().strip(),
        ],
        required=True,
    )
    python = source / ".venv/bin/python"
    run(
        "resolved-dependencies",
        [sys.executable, "-m", "uv", "pip", "freeze", "--python", str(python)],
    )
    run("gpu-inventory", ["nvidia-smi", "-q"])

    target_code = "import json,sys,torch; from pathlib import Path; from pagedcore.cli import collect_environment_diagnostics; assert torch.cuda.is_available() and torch.cuda.device_count()==1; assert 'T4' in torch.cuda.get_device_name(0); assert torch.__version__=='2.7.1+cu126'; p=torch.cuda.get_device_properties(0); Path(sys.argv[1]).write_text(json.dumps(dict(torch=torch.__version__,name=p.name,uuid=str(p.uuid),capability=[p.major,p.minor],visible_devices=torch.cuda.device_count(),total_memory=p.total_memory,environment=collect_environment_diagnostics()),indent=2))"
    run("target", [str(python), "-c", target_code, str(evidence / "target.json")], required=True)
    run(
        "checkpoint-verify",
        [
            str(python),
            "-m",
            "scripts.verify_owned_checkpoint",
            "--output",
            str(evidence / "environment-manifest.json"),
        ],
        required=True,
    )
    setup_ok = True
except BaseException:
    (evidence / "setup-failure.txt").write_text(traceback.format_exc())
    print("Setup incomplete. Run the final cell to download its logs.")

In [ ]:
reproduction_ok = False
try:
    assert setup_ok, "Setup did not complete"
    run(
        "reproduction",
        [
            str(python),
            "-m",
            "scripts.probe_two_token_decode",
            "--prompt-tokens",
            "15",
            "--single-only",
            "--output-dir",
            str(evidence / "reproduction"),
        ],
        required=True,
    )
    exact_check = """
import json, sys, torch
from pathlib import Path
root, output = map(Path, sys.argv[1:])
prior = torch.load(root / 'prior-failure/target.pt', map_location='cpu', weights_only=True)
current = torch.load(output / 'reproduction/single/target.pt', map_location='cpu', weights_only=True)
controls = {name + '_exact': torch.equal(prior[name], current[name]) for name in prior}
(output / 'prior-failure-controls.json').write_text(json.dumps(controls, indent=2))
assert all(controls.values()), 'Raw logits differ from retained T4 failure; inspect environment/source'
"""
    run(
        "prior-failure-control",
        [str(python), "-c", exact_check, str(source), str(evidence)],
        required=True,
    )
    result = json.loads((evidence / "reproduction/single/report.json").read_text())
    print("Attended tokens:", result["attended_tokens"])
    print(
        "Logit tolerance ratio:",
        result["owned_reference"]["max_tolerance_ratio"],
        "(above 1 fails)",
    )
    print("Greedy choice agrees:", result["owned_reference"]["greedy_matches"])
    assert result["discrepancy_reproduced"], "The known failure did not reproduce"
    reproduction_ok = True
except BaseException:
    (evidence / "reproduction-failure.txt").write_text(traceback.format_exc())
    print("Reproduction incomplete. Run the final cell to download its logs.")

In [ ]:
try:
    assert reproduction_ok, "The retained failure must reproduce first"
    run(
        "scalar-control",
        [
            str(python),
            "-m",
            "scripts.intervene_decode_scalar",
            "--baseline-dir",
            str(evidence / "reproduction/single"),
            "--trace-dir",
            str(source / "prior-trace"),
            "--output-dir",
            str(evidence / "scalar-control"),
        ],
        required=True,
    )
    result = json.loads((evidence / "scalar-control/report.json").read_text())
    print("Control valid:", result["valid_control"])
    print("Baseline/trace checks:", result["controls"])
    for name, arm in result["arms"].items():
        print(
            name,
            "logit ratio:",
            arm["logits"]["max_tolerance_ratio"],
            "failing logits:",
            arm["failing_logit_elements"],
            "K/V ratio:",
            arm["trace"]["kv_measurement"]["max_kv_tolerance_ratio"],
        )
    print(
        "Target layer output now matches reference:",
        result["arms"]["single_scalar"]["target_hidden_out_equals_reference"],
    )
    print("First remaining FP16 difference:", result["remaining_first_fp16_difference"])
except BaseException:
    (evidence / "scalar-control-failure.txt").write_text(traceback.format_exc())
    print("Control incomplete. Run the final cell to download partial evidence.")

In [ ]:
try:
    (evidence / "source-after.json").write_text(json.dumps(verify_source(), indent=2))
except BaseException:
    (evidence / "source-check-failure.txt").write_text(traceback.format_exc())
(evidence / "commands.json").write_text(json.dumps(commands, indent=2))
(evidence / "notebook-run.json").write_text(
    json.dumps(
        {
            "snapshot_sha256": SNAPSHOT_SHA256,
            "base_revision": identity["base_revision"],
            "scope": "Single-scalar intervention; causal diagnostic, never a serving repair or acceptance",
            "environment_overrides": {
                k: environment[k]
                for k in (
                    "CUDA_VISIBLE_DEVICES",
                    "CUBLAS_WORKSPACE_CONFIG",
                    "OMP_NUM_THREADS",
                    "MKL_NUM_THREADS",
                )
            },
        },
        indent=2,
    )
)
files = {
    p.relative_to(evidence).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
    for p in evidence.rglob("*")
    if p.is_file()
}
(evidence / "checksums.json").write_text(json.dumps(files, indent=2))
archive_path = working / (run_root.name + ".zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("source-snapshot.zip", payload)
    for p in evidence.rglob("*"):
        if p.is_file():
            archive.write(p, "evidence/" + p.relative_to(evidence).as_posix())
print("Download and send back:", archive_path.name)
print("If the link fails, download the ZIP from Kaggle's Output file panel.")
display(
    FileLink(
        str(archive_path.relative_to(Path.cwd()))
        if archive_path.is_relative_to(Path.cwd())
        else str(archive_path)
    )
)